In [1]:
import dspy
from pydantic import BaseModel, Field
from typing import List

# Konfiguration des LLM
local_llm = dspy.LM(
    "openai/gemma3:4b", 
    api_base="http://localhost:11434/v1", 
    api_key="no_key_needed"
)

dspy.configure(lm=local_llm,  cache=False)
dspy.configure_cache(enable_disk_cache=False)
dspy.configure_cache(enable_memory_cache=False)

In [2]:
# 1. Definition des Ziel-Datenmodells
class ProductSchema(BaseModel):
    name: str = Field(..., description="Der genaue Name des Produkts.")
    price: float = Field(..., description="Der Preis des Produkts als Zahl.")
    features: List[str] = Field(..., description="Eine Liste der wichtigsten technischen Merkmale.")


In [3]:
# 2. Definition der Signatur
class ProductExtraction(dspy.Signature):
    """
    Analysiert eine Produktbeschreibung und extrahiert strukturierte Daten gemäß dem definierten Schema.
    """
    description: str = dspy.InputField(desc="Der unstrukturierte Werbetext des Produkts.")
    product_data: ProductSchema = dspy.OutputField(desc="Die extrahierten Produktinformationen als Objekt auf deutsch.")



In [4]:
# 3. Das Extraktions-Modul (Standard Predictor)
class ProductExtractor(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extractor = dspy.Predict(ProductExtraction)

    def forward(self, description):
        return self.extractor(description=description)

In [5]:
# 4. Die "Assertion"-Logik als Validierungsfunktion
# Diese Funktion ersetzt dspy.Assert. Sie gibt True zurück, wenn alles okay ist, sonst False.
def validate_product_data(example, pred, trace=None):
    try:
        # Zugriff auf das extrahierte Objekt
        data = pred.product_data
        
        # Die eigentliche Constraint-Prüfung:
        # 1. Existiert das Objekt?
        if not data:
            return False
        # 2. Ist der Preis ein Float/Int und positiv?
        if not isinstance(data.price, (float, int)) or data.price <= 0:
            return False
            
        return True
    except Exception:
        return False


In [6]:
# 5. Verwendung von dspy.Refine statt Retry/Assert
# dspy.Refine wickelt das Modul ein. Wenn 'reward_fn' False zurückgibt (also die Assertion fehlschlägt),
# versucht Refine es erneut (bis zu N mal).
extractor_module = ProductExtractor()
extractor = dspy.Refine(
    module=extractor_module,
    N=3,  # Maximale Versuche (Retries)
    reward_fn=validate_product_data, # Unsere Validierungsfunktion
    threshold = 1.0
)

In [7]:
# 6. Ausführung
raw_text = """
Hol dir das Santa Cruz Classic Dot 80s Cruiser Skateboard für dein nächstes Abenteuer. 
Dieses Board kostet aktuell 149,95 Euro und bietet echtes Retro-Feeling. 
Es besteht aus 7-lagigem nordamerikanischem Ahorn und ist mit weichen 
60mm Slime Balls Rollen ausgestattet, die perfekt für rauen Asphalt sind. 
Zudem verfügt es über hochwertige Krux Achsen und das ikonische Logo-Design auf der Unterseite.
"""

print("Starte Extraktion mit Refine-Logik...")

# Der Aufruf sieht aus wie immer, aber im Hintergrund läuft jetzt die Validierungsschleife
try:
    response = extractor(description=raw_text)
    
    # Zugriff auf das Ergebnis (Refine gibt das beste gefundene Ergebnis zurück)
    extracted_data = response.product_data
    
    print("-" * 30)
    print(extracted_data.model_dump_json(indent=2))
    print("-" * 30)
    print(f"Preis erfolgreich validiert: {extracted_data.price} €")

except Exception as e:
    print(f"Fehler bei der Extraktion: {e}")



Starte Extraktion mit Refine-Logik...
------------------------------
{
  "name": "Santa Cruz Classic Dot 80s Cruiser Skateboard",
  "price": 149.95,
  "features": [
    "7-lagiges nordamerikanisches Ahorn",
    "weiche 60mm Slime Balls Rollen",
    "Hochwertige Krux Achsen",
    "Ikonisches Logo-Design"
  ]
}
------------------------------
Preis erfolgreich validiert: 149.95 €


In [8]:
# Optional: Inspektion der Gedankengänge
local_llm.inspect_history(n=10)